In [ ]:
import json
import os
import numpy as np
import math
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches

from PIL import Image
from pathlib import Path
from collections import defaultdict, Counter

### 0. Load data

In [ ]:
ANNOTATIONS_PATH = Path("../data/ClearSAR/data/annotations/instances_train.json")

TRAIN_SET_PATH = Path("../data/ClearSAR/data/images/train")
TEST_SET_PATH = Path("../data/ClearSAR/data/images/test")

In [ ]:
with open(ANNOTATIONS_PATH) as f:
      annotations = json.load(f)

In [ ]:
images_dict = {img["id"]: img for img in annotations["images"]}

anns_by_image = defaultdict(list)
for ann in annotations["annotations"]:
    anns_by_image[ann["image_id"]].append(ann)

### 1. Image size distribution check

In [ ]:
images_dict[1260]['width']

In [ ]:
widths = [images_dict[i]["width"] for i in images_dict]
heights = [images_dict[i]["height"] for i in images_dict]

In [ ]:
fig, axs = plt.subplots(1, 2)

axs[0].hist(widths, bins=30, label="widths")
axs[0].set_title("Widths")
axs[1].hist(heights, bins=30, label="heights")
axs[1].set_title("Heights")
plt.show()

In [ ]:
size_counts = Counter((img["width"], img["height"]) for img in images_dict.values())
print(size_counts)

**Findings**
- Widths are clustered between ~500–535px, no outliers
- Heights are mainly ~340–345px, but the histogram tail and the counter reveal some outliers

### 2. Test set size check

In [ ]:
test_images = []
for i in os.listdir(TEST_SET_PATH):
      img_path = os.path.join(TEST_SET_PATH, i)
      test_images.append(img_path)

In [ ]:
test_widths = [Image.open(i).size[0] for i in test_images]
test_heights = [Image.open(i).size[1] for i in test_images]

test_size_counts = Counter(zip(test_widths, test_heights))
print(test_size_counts)

In [ ]:
fig, axs = plt.subplots(1, 2)
fig.set_figheight(5)
fig.set_figwidth(10)

axs[0].hist(widths, bins=30, label="widths", color='green', alpha=0.5)
axs[0].hist(test_widths, bins=30, label="widths", color='red', alpha=0.5)
axs[0].set_title("Widths (Green=train, Red=test)")
axs[1].hist(heights, bins=30, label="heights", color='green', alpha=0.5)
axs[1].hist(test_heights, bins=30, label="heights", color="red", alpha=0.5)
axs[1].set_title("Heights (Green=train, Red=test)")
plt.show()

**Findings**
- Widths of the test set have the same 500–535px range, and same distribution shape
- Heights are mostly 340–345px
- Test set also has a few "tall" images 

### 3. Visual inspection

In [ ]:
def show_examples(image_ids, cols=3):
      rows = math.ceil(len(image_ids) / cols)
      fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 3))
      axes = axes.flatten()
      
      for i, img_id in enumerate(image_ids):
            img_info = images_dict[img_id]
            img_path = os.path.join(TRAIN_SET_PATH, img_info["file_name"])
            img = np.array(Image.open(img_path))
            
            axes[i].imshow(img)
            for ann in anns_by_image[img_id]:
                  x, y, w, h = ann["bbox"]
                  rect = patches.Rectangle((x, y), w, h, linewidth=1, edgecolor="red", 
                                          fill=False)
                  axes[i].add_patch(rect)
            axes[i].set_title(f"id={img_id}, n={len(anns_by_image[img_id])}")
            axes[i].axis("off")
      
      for j in range(i + 1, len(axes)):
            axes[j].axis("off")
      
      plt.tight_layout()
      plt.show()

In [ ]:
# extract "typical" IDs, i.e. where annotation counts is close to 3 (the median)
typical_ids = [img_id for img_id in anns_by_image if len(anns_by_image[img_id]) == 3]

show_examples(random.sample(typical_ids, 12))

**Findings**
- There are some thin horizontal boxes, with RFI patterns that are easy to spot
- There are some large boxes as well, probably a different type of interference that affects a whole region, although visually there is not anything odd that can be visualized.

In [ ]:
# let's also check some "dense" cases
dense_ids = sorted(anns_by_image, key=lambda x: len(anns_by_image[x]), reverse=True)[:12]

show_examples(dense_ids)

**Findings**:
- The "classic" thin horizontal detections are very clear here
- Many images show stacked parallel RFI streaks at the same X position but at different heights

In [ ]:
# let's also investigate some edge cases

# group 1: tiny boxes
tiny_ids = list(set(ann["image_id"] for ann in annotations["annotations"] if ann["bbox"][3] == 1))

# group 2: big coverage of boxes
big_ids = list(set(ann["image_id"] for ann in annotations["annotations"] 
                  if ann["area"] / (images_dict[ann["image_id"]]["width"] * images_dict[ann["image_id"]]["height"]) >= 0.66))

# group 3: no RFIs
neg_ids = [img["id"] for img in annotations["images"] if img["id"] not in anns_by_image]

In [ ]:
for i in [tiny_ids, big_ids, neg_ids]:
      show_examples(i)

**Findings**:
- The negative images show some suspicious examples, e.g. it seems annotators may have missed (or not annotated on purpose) some artifacts that really look like RFIs.